<a href="https://colab.research.google.com/github/rayendhawadi/Ai-agent/blob/main/Ai_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U langchain-google-genai langgraph

In [ ]:
!pip install -U langchain-groq langgraph tavily-python langchain-community -q

In [ ]:
!pip install langchain-groq

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')

In [ ]:
import json
from typing import TypedDict, Optional
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3
)

print("✅ Modèle chargé : llama-3.3-70b-versatile (Groq)")

✅ Modèle chargé : llama-3.3-70b-versatile (Groq)


In [ ]:
class AgentState(TypedDict):
    question: str
    plan: str
    research_notes: str
    draft: str
    critique: str
    approved: bool        # ✨ le critic décide si c'est bon
    iteration: int
    max_iterations: int

In [ ]:
# ============================================================
# 🗺️  PLANNER
# ============================================================
PLANNER_PROMPT = """Tu es un expert en planification de rapports analytiques.

Ton rôle : lire la question de l'utilisateur et produire un plan de rapport clair et structuré.

Format attendu :
- Entre 3 et 5 sections numérotées
- Chaque section a un titre court et 2-3 sous-points
- Longueur totale : 150-200 mots maximum
- Style : professionnel, concis, sans introduction ni conclusion

Réponds UNIQUEMENT avec le plan, sans commentaire supplémentaire."""

def planner_node(state: AgentState):
    msg = llm.invoke([
        SystemMessage(content=PLANNER_PROMPT),
        HumanMessage(content=f"Question : {state['question']}")
    ])
    return {"plan": msg.content}

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

# Outil de recherche web réelle
search_tool = TavilySearchResults(
    max_results=4,
    search_depth="advanced"
)

RESEARCHER_PROMPT = """Tu es un expert en recherche et synthèse d'informations.

Ton rôle : synthétiser des résultats de recherche web en notes structurées et factuelles.

Instructions :
- Garde uniquement les informations pertinentes et fiables
- Cite la source entre parenthèses à la fin de chaque fait : (source : url)
- Organise par section avec des puces (•)
- Longueur : 80-120 mots par section
- Style : factuel, précis, sans opinions personnelles
- Élimine les doublons entre les sources

Réponds UNIQUEMENT avec les notes de recherche structurées."""

def researcher_node(state: AgentState):
    # 1. Extraire les titres de sections numérotées du plan
    sections = state['plan'].split('\n')
    queries = [line.strip() for line in sections if line.strip() and line[0].isdigit()]

    # 2. Recherche web pour chaque section (max 3 pour économiser le quota)
    all_results = []
    for query in queries[:3]:
        try:
            results = search_tool.invoke(query + " " + state['question'])
            for r in results:
                all_results.append(f"- {r['content']} (source : {r['url']})")
        except Exception as e:
            all_results.append(f"- Recherche échouée pour '{query}': {str(e)}")

    web_content = "\n".join(all_results)

    # 3. Le LLM synthétise les vrais résultats web
    msg = llm.invoke([
        SystemMessage(content=RESEARCHER_PROMPT),
        HumanMessage(content=f"Plan :\n{state['plan']}\n\nRésultats de recherche web :\n{web_content}")
    ])

    return {"research_notes": msg.content}

In [ ]:
# ============================================================
# ✍️  WRITER
# ============================================================
WRITER_PROMPT = """Tu es un rédacteur technique senior spécialisé dans les rapports professionnels.

Ton rôle : transformer des notes de recherche en un rapport fluide, complet et bien structuré.

Format attendu :
- Une introduction courte (2-3 phrases) qui pose le contexte
- Des sections avec titres en gras (**Titre**)
- Des paragraphes de 4-6 phrases, rédigés en prose (pas de listes à puces)
- Une conclusion synthétique (3-4 phrases)
- Longueur totale : 400-600 mots
- Style : formel, clair, sans jargon inutile

Si une critique t'est fournie, intègre-la point par point dans ta nouvelle version.
Réponds UNIQUEMENT avec le rapport rédigé."""

def writer_node(state: AgentState):
    prompt = f"Notes de recherche :\n{state['research_notes']}"
    if state.get('critique'):
        prompt += f"\n\n---\nCritique à intégrer dans cette nouvelle version :\n{state['critique']}"
    msg = llm.invoke([
        SystemMessage(content=WRITER_PROMPT),
        HumanMessage(content=prompt)
    ])
    return {
        "draft": msg.content,
        "iteration": state.get("iteration", 0) + 1
    }


# ============================================================
# 🎯  CRITIC — décide lui-même d'approuver ou non
# ============================================================
CRITIC_PROMPT = """Tu es un critique éditorial exigeant dont le rôle est d'évaluer la qualité d'un rapport.

Tu dois évaluer le rapport sur ces critères :
1. Clarté et lisibilité
2. Précision et profondeur des informations
3. Structure et cohérence
4. Qualité de la conclusion

⚠️ IMPORTANT — Tu dois terminer ta réponse par exactement une de ces deux lignes :
  VERDICT: APPROUVÉ   (si le rapport est de bonne qualité, publiable tel quel)
  VERDICT: À AMÉLIORER  (si des corrections importantes sont nécessaires)

Format de ta réponse :
- Points positifs : (2-3 points)
- Points à améliorer : (2-3 points précis et actionnables)
- VERDICT: APPROUVÉ ou VERDICT: À AMÉLIORER

Sois honnête et constructif. Si le rapport est bon, dis-le clairement."""

def critic_node(state: AgentState):
    msg = llm.invoke([
        SystemMessage(content=CRITIC_PROMPT),
        HumanMessage(content=f"Rapport à évaluer (itération {state.get('iteration', 1)}) :\n\n{state['draft']}")
    ])
    critique_text = msg.content

    # ✨ On lit le verdict du critic
    approved = "VERDICT: APPROUVÉ" in critique_text.upper()

    return {
        "critique": critique_text,
        "approved": approved
    }

In [ ]:
def should_continue(state: AgentState):
    # Sécurité : limite max
    if state["iteration"] >= state["max_iterations"]:
        print(f"⏹️  Limite d'itérations atteinte ({state['max_iterations']})")
        return "end"

    # ✨ Le critic a approuvé ? On arrête
    if state.get("approved", False):
        print("✅ Rapport approuvé par le critic !")
        return "end"

    print(f"🔄 Itération {state['iteration']} — Le critic demande des améliorations...")
    return "continue"

In [ ]:
workflow = StateGraph(AgentState)

workflow.add_node("planner",    planner_node)
workflow.add_node("researcher", researcher_node)
workflow.add_node("writer",     writer_node)
workflow.add_node("critic",     critic_node)

workflow.set_entry_point("planner")
workflow.add_edge("planner",    "researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer",     "critic")

workflow.add_conditional_edges(
    "critic",
    should_continue,
    {
        "continue": "writer",
        "end": END
    }
)

app = workflow.compile()
print("✅ Graphe compilé avec succès !")


✅ Graphe compilé avec succès !


In [ ]:
STEP_LABELS = {
    "planner":    ("🗺️ ", "PLANNER",    "plan"),
    "researcher": ("🔬",  "RESEARCHER", "research_notes"),
    "writer":     ("✍️ ", "WRITER",     "draft"),
    "critic":     ("🎯",  "CRITIC",     "critique"),
}

def display_step(step_name, output):
    if step_name not in STEP_LABELS:
        return
    emoji, label, content_key = STEP_LABELS[step_name]
    content = output.get(content_key, "")
    sep = "═" * 60
    print(f"\n{sep}")
    print(f"  {emoji}  {label}")
    print(sep)
    print(content)

# ▶️ Lancement
inputs = {
    "question": "Comment l'IA peut-elle optimiser la gestion du réseau électrique ?",
    "max_iterations": 5,
    "iteration": 0,
    "plan": "",
    "research_notes": "",
    "draft": "",
    "critique": "",
    "approved": False
}

print("🚀 Démarrage du pipeline multi-agents...")
print(f"   Question : {inputs['question']}\n")

for output in app.stream(inputs):
    for step_name, step_output in output.items():
        display_step(step_name, step_output)

print("\n" + "═" * 60)
print("  ✅  PIPELINE TERMINÉ")
print("═" * 60)



🚀 Démarrage du pipeline multi-agents...
   Question : Comment l'IA peut-elle optimiser la gestion du réseau électrique ?


════════════════════════════════════════════════════════════
  🗺️   PLANNER
════════════════════════════════════════════════════════════
1. Introduction de l'IA
 * Utilisation de l'apprentissage automatique pour analyser les données de réseau
 * Intégration de l'IA dans les systèmes de gestion de réseau existants

2. Prévisions et optimisation
 * Prévisions de la demande d'électricité en temps réel
 * Optimisation de la production et de la distribution d'énergie

3. Sécurité et maintenance
 * Détection précoce des anomalies et des pannes potentielles
 * Planification de la maintenance préventive pour réduire les temps d'arrêt

════════════════════════════════════════════════════════════
  🔬  RESEARCHER
════════════════════════════════════════════════════════════
## Introduction de l'IA
* L'IA est utilisée pour analyser les données de réseau et optimiser les système

In [ ]:
final = app.invoke(inputs)

print("═" * 60)
print(f"  📄  RAPPORT FINAL  (itération {final['iteration']})")
print("═" * 60)
print(final["draft"])
print("\n" + "─" * 60)
print(f"  Approuvé par le critic : {'✅ OUI' if final['approved'] else '⚠️  NON (limite atteinte)'}")
print(f"  Nombre d'itérations    : {final['iteration']}")

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kczy9qrtests6k9dt3ywf701` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 98635, Requested 6006. Please try again in 1h6m49.824s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
import gradio as gr

# ============================================================
# 🎨 Fonction principale appelée par Gradio
# ============================================================
def run_pipeline(question, max_iterations):
    inputs = {
        "question": question,
        "max_iterations": int(max_iterations),
        "iteration": 0,
        "plan": "",
        "research_notes": "",
        "draft": "",
        "critique": "",
        "approved": False
    }

    log = ""
    plan_out = ""
    research_out = ""
    draft_out = ""
    critique_out = ""

    # Parcourt chaque étape du pipeline et met à jour l'affichage
    for output in app.stream(inputs):
        for step_name, step_output in output.items():

            if step_name == "planner":
                plan_out = step_output.get("plan", "")
                log += "✅ Planner terminé\n"

            elif step_name == "researcher":
                research_out = step_output.get("research_notes", "")
                log += "✅ Researcher terminé\n"

            elif step_name == "writer":
                draft_out = step_output.get("draft", "")
                iteration = step_output.get("iteration", "?")
                log += f"✅ Writer terminé — itération {iteration}\n"

            elif step_name == "critic":
                critique_out = step_output.get("critique", "")
                approved = step_output.get("approved", False)
                if approved:
                    log += "🎉 Critic : APPROUVÉ !\n"
                else:
                    log += "🔄 Critic : demande des améliorations...\n"

    return plan_out, research_out, draft_out, critique_out, log


# ============================================================
# 🖥️ Construction de l'interface
# ============================================================
with gr.Blocks(theme=gr.themes.Soft(), title="Agent Multi-Rôles") as demo:

    gr.Markdown("""
    # 🤖 Agent Multi-Rôles — LangGraph + Groq
    Pose une question, l'agent va **planifier → rechercher → rédiger → critiquer** automatiquement.
    """)

    # --- Inputs ---
    with gr.Row():
        with gr.Column(scale=4):
            question_input = gr.Textbox(
                label="💬 Ta question",
                placeholder="Ex: Comment l'IA peut-elle optimiser la gestion du réseau électrique ?",
                lines=2
            )
        with gr.Column(scale=1):
            iterations_input = gr.Slider(
                label="🔁 Max itérations",
                minimum=1,
                maximum=5,
                value=3,
                step=1
            )

    run_btn = gr.Button("🚀 Lancer le pipeline", variant="primary", size="lg")

    # --- Log de progression ---
    log_output = gr.Textbox(
        label="📋 Progression",
        lines=5,
        interactive=False
    )

    # --- Outputs par onglet ---
    with gr.Tabs():
        with gr.Tab("🗺️ Plan"):
            plan_output = gr.Markdown(label="Plan structuré")

        with gr.Tab("🔬 Notes de recherche"):
            research_output = gr.Markdown(label="Notes du Researcher")

        with gr.Tab("✍️ Rapport final"):
            draft_output = gr.Markdown(label="Rapport rédigé")

        with gr.Tab("🎯 Critique"):
            critique_output = gr.Markdown(label="Évaluation du Critic")

    # --- Connexion bouton → fonction ---
    run_btn.click(
        fn=run_pipeline,
        inputs=[question_input, iterations_input],
        outputs=[plan_output, research_output, draft_output, critique_output, log_output]
    )

# Lancement
demo.launch(debug=True)

/tmp/ipykernel_8376/897169212.py:55: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Agent Multi-Rôles") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7f1061a1f1ae31eaac.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1698, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 63, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^